<a href="https://colab.research.google.com/github/AzizulHakim00/Glaucomma/blob/main/Glaucomma_RimGraphDG_V45_ORIGA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RimGraph-DG V4.5 — mask-audited ORIGA-first run
Use a **T4 GPU**. This run validates decoded OD/OC supervision before model training, reuses the completed V4.4 ORIGA baseline only if split/config signatures match exactly, and trains RimGraph fresh.

In [1]:
GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': '',
    'fast_dev_run': False,
    'run_name': 'paper_run_v45',
    'code_revision': 'rimgraph-dg-v4.5-20260809',
    'seeds': [2029],
    'fold_targets': ['ORIGA'],
    'run_global_baseline': True,
    'run_full_model': True,
    'baseline_reuse_run': 'paper_run_v44',
    'run_optuna': False,
    'image_size': 320,
    'batch_size': 2,
    'grad_accum': 2,
    'fpn_dim': 128,
    'gradient_checkpointing': True,
    'baseline_epochs': 12,
    'full_epochs': 30,
    'seg_warmup_epochs': 5,
    'anatomy_warmup_epochs': 10,
    'num_workers': 0,
    'n_visual_examples': 2,
    'resume': True,
}

import hashlib, json, traceback, urllib.request
from pathlib import Path
import torch

print('=== V4.5 LAUNCHER GPU CHECK ===', flush=True)
print('PyTorch:', torch.__version__, flush=True)
print('CUDA available:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('T4 GPU is not active. Colab: Runtime > Change runtime type > T4 GPU, reconnect, then rerun.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('================================', flush=True)

COMMIT = 'd71c11be1a13a25ed0352cdde01453c17b234429'
EXPECTED_RAW_SHA256 = '46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2'
ROOT = f'https://raw.githubusercontent.com/AzizulHakim00/Glaucomma/{COMMIT}'
parts = [f'v4_parts/part_{i:02d}.py' for i in range(7)]
raw_code = '\n'.join(urllib.request.urlopen(f'{ROOT}/{name}').read().decode('utf-8') for name in parts)
actual_raw = hashlib.sha256(raw_code.encode('utf-8')).hexdigest()
assert actual_raw == EXPECTED_RAW_SHA256, f'Raw runner integrity check failed: {actual_raw}'

patch_specs = [
    ('runner_patch_v41.py', 'apply_v41'),
    ('runner_patch_v42.py', 'apply_v42'),
    ('runner_patch_v43.py', 'apply_v43'),
    ('runner_patch_v43_autograd.py', 'apply_v43_autograd'),
    ('runner_patch_v44_runtime.py', 'apply_v44_runtime'),
    ('runner_patch_v45_masks.py', 'apply_v45_masks'),
    ('runner_patch_v45_lowlabels.py', 'apply_v45_lowlabels'),
]
code = raw_code
for patch_name, function_name in patch_specs:
    print(f'[LAUNCHER] applying {patch_name}', flush=True)
    source = urllib.request.urlopen(f'{ROOT}/{patch_name}').read().decode('utf-8')
    namespace = {}
    exec(compile(source, patch_name, 'exec'), namespace, namespace)
    code = namespace[function_name](code)
compile(code, 'rimgraph_dg_v45_origa.py', 'exec')
print('[LAUNCHER] V4.5 assembly PASSED', flush=True)

try:
    exec(code, globals(), globals())
    completion = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45/RUN_COMPLETED.json')
    if not completion.exists():
        raise RuntimeError('Runner returned without RUN_COMPLETED.json; treating this as failure.')
    print('\n✅ V4.5 VERIFIED COMPLETION:', completion, flush=True)
except BaseException:
    trace = traceback.format_exc()
    print('\n=== V4.5 FAILURE TRACEBACK ===', flush=True)
    print(trace, flush=True)
    try:
        Path('/content/RimGraph_V45_FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
    except Exception:
        pass
    try:
        failure_dir = Path('/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45')
        failure_dir.mkdir(parents=True, exist_ok=True)
        (failure_dir / 'FAILURE_TRACEBACK.txt').write_text(trace, encoding='utf-8')
        (failure_dir / 'FAILURE_STATUS.json').write_text(json.dumps({'status': 'failed', 'traceback_file': str(failure_dir / 'FAILURE_TRACEBACK.txt')}, indent=2), encoding='utf-8')
    except Exception as drive_error:
        print(f'Could not persist failure to Drive: {drive_error}', flush=True)
    raise


=== V4.5 LAUNCHER GPU CHECK ===
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
[LAUNCHER] applying runner_patch_v41.py
[LAUNCHER] applying runner_patch_v42.py
[LAUNCHER] applying runner_patch_v43.py
[LAUNCHER] applying runner_patch_v43_autograd.py
[LAUNCHER] applying runner_patch_v44_runtime.py
[LAUNCHER] applying runner_patch_v45_masks.py
[LAUNCHER] applying runner_patch_v45_lowlabels.py
[LAUNCHER] V4.5 assembly PASSED
Mounted at /content/drive
Resolved Colab output: /content/Glaucomma_runs/paper_run_v45
Resolved Drive output: /content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45
Drive write verification: PASSED

=== RIMGRAPH V4.4 RUNTIME ===
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB



## RimGraph-DG V4 configuration

,setting,value
0,kaggle_dataset,arnavjain1/glaucoma-datasets
1,manual_data_dir,
2,sources,"['ORIGA', 'REFUGE', 'G1020']"
3,fold_targets,['ORIGA']
4,image_size,320
5,num_workers,0
6,canonicalize_laterality,True
7,exclude_cross_source_duplicates,True
8,project_name,RimGraph_DG_V4
9,run_name,paper_run_v45


## Downloading or locating Kaggle dataset

100%|██████████| 5.55G/5.55G [04:57<00:00, 20.1MB/s]

...

,auroc,auprc,accuracy,balanced_accuracy,sensitivity,specificity,sensitivity_at_95_specificity,precision,f1,mcc,...,calibration_method,temperature,n_test,best_epoch,dice_disc,iou_disc,n_mask_disc,dice_cup,iou_cup,n_mask_cup
0,0.7627,0.5216,0.5938,0.6757,0.8452,0.5062,0.2798,0.3737,0.5182,0.3122,...,robust_temperature,5.3345,650,5,NaN,NaN,NaN,NaN,NaN,NaN
1,0.7765,0.4926,0.7538,0.6789,0.5238,0.8340,0.1786,0.5238,0.5238,0.3578,...,robust_temperature,4.5514,650,13,0.9495,0.905,650.0,0.7343,0.6405,650.0


## Baseline versus full model

,held_out_source,seed,baseline_auroc,full_auroc,delta_auroc,baseline_auprc,full_auprc,delta_auprc
0,ORIGA,2029,0.7627,0.7765,0.0138,0.5216,0.4926,-0.0289


# RimGraph-DG V4 Scientific Diagnostic Report

- Global baseline mean external AUROC: **0.7627**
- Global baseline worst-domain AUROC: **0.7627**
- RimGraph V4 mean external AUROC: **0.7765**
- RimGraph V4 worst-domain AUROC: **0.7765**
- Mean disc Dice: **0.9495**
- Mean cup Dice: **0.7343**
- Mean ECE: **0.1221**

## Interpretation
- Full-model discrimination is close to the baseline; novelty must be justified through calibration, robustness, segmentation, and ablations rather than AUROC alone.
- Cup segmentation is too weak for reliable structural graph interpretation. Improve masks/decoder before publication claims.
- Severe domain-generalization weakness remains; the current result is not journal-ready.
- Calibration is poor; probabilities should not be presented as clinically reliable.

## Required publication checks
- Run at least three seeds for every held-out domain.
- Report bootstrap confidence intervals and baseline-versus-full deltas.
- Polar-sector XAI is image-oriented when laterality is unknown; do not label it as clinical clock-hour anatomy.
- gate_context is a latent fusion variable, not a supervised image-quality score.
- Manually inspect segmentation and XAI failures.
- Add one completely untouched external dataset for a stronger Q1/Q2 claim.


✅ V4.5 VERIFIED COMPLETION: /content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45/RUN_COMPLETED.json


In [2]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path(
    "/content/drive/MyDrive/Glaucomma_RimGraphDG/"
    "paper_run_v45/folds/ORIGA/seed_2029"
)

OUT = Path(
    "/content/drive/MyDrive/Glaucomma_RimGraphDG/"
    "paper_run_v45/EXTRACTED_FINAL_REPORT"
)
OUT.mkdir(parents=True, exist_ok=True)

models = {
    "Global Baseline": ROOT / "global_baseline",
    "RimGraph-DG V4.5": ROOT / "rimgraph_v4",
}

# ============================================================
# 1. EXACT METRICS.JSON
# ============================================================

rows = []

for name, path in models.items():
    metrics_file = path / "metrics.json"

    if not metrics_file.exists():
        raise FileNotFoundError(metrics_file)

    with open(metrics_file, "r") as f:
        m = json.load(f)

    m["display_name"] = name
    rows.append(m)

metrics = pd.DataFrame(rows)

preferred = [
    "display_name",
    "model_type",
    "held_out_source",
    "seed",
    "n_test",

    "auroc",
    "auroc_ci_low",
    "auroc_ci_high",

    "auprc",
    "auprc_ci_low",
    "auprc_ci_high",

    "accuracy",
    "balanced_accuracy",

    "sensitivity",
    "sensitivity_ci_low",
    "sensitivity_ci_high",

    "specificity",
    "specificity_ci_low",
    "specificity_ci_high",

    "sensitivity_at_95_specificity",
    "precision",
    "f1",
    "mcc",

    "brier",
    "nll",
    "ece",

    "threshold",

    "tn",
    "fp",
    "fn",
    "tp",

    "calibration_method",
    "temperature",
    "best_epoch",

    "dice_disc",
    "iou_disc",
    "n_mask_disc",

    "dice_cup",
    "iou_cup",
    "n_mask_cup",
]

preferred = [c for c in preferred if c in metrics.columns]

remaining = [
    c for c in metrics.columns
    if c not in preferred
]

metrics = metrics[preferred + remaining]


# ============================================================
# 2. CALIBRATION SELECTION
# ============================================================

calibration = {}

for name, path in models.items():
    f = path / "calibration_selection.csv"

    if f.exists():
        df = pd.read_csv(f)
        df.insert(0, "model", name)
        calibration[name] = df


# ============================================================
# 3. FULL TRAINING HISTORIES
# ============================================================

histories = {}

for name, path in models.items():
    f = path / "history.csv"

    if f.exists():
        df = pd.read_csv(f)
        df.insert(0, "model", name)
        histories[name] = df


# ============================================================
# 4. CONFUSION MATRIX TABLE
# ============================================================

conf_cols = [
    c for c in
    [
        "display_name",
        "tn", "fp", "fn", "tp",
        "sensitivity",
        "specificity",
        "precision",
        "f1",
        "mcc"
    ]
    if c in metrics.columns
]

confusion = metrics[conf_cols].copy()


# ============================================================
# 5. BOOTSTRAP CI TABLE
# ============================================================

ci_cols = [
    c for c in [
        "display_name",

        "auroc",
        "auroc_ci_low",
        "auroc_ci_high",

        "auprc",
        "auprc_ci_low",
        "auprc_ci_high",

        "sensitivity",
        "sensitivity_ci_low",
        "sensitivity_ci_high",

        "specificity",
        "specificity_ci_low",
        "specificity_ci_high",
    ]
    if c in metrics.columns
]

ci_table = metrics[ci_cols].copy()


# ============================================================
# 6. BASELINE → RIMGRAPH DELTAS
# ============================================================

delta_metrics = [
    "auroc",
    "auprc",
    "accuracy",
    "balanced_accuracy",
    "sensitivity",
    "specificity",
    "precision",
    "f1",
    "mcc",
    "brier",
    "nll",
    "ece",
]

delta_rows = []

if len(metrics) == 2:

    base = metrics[
        metrics["display_name"] == "Global Baseline"
    ].iloc[0]

    full = metrics[
        metrics["display_name"] == "RimGraph-DG V4.5"
    ].iloc[0]

    for metric in delta_metrics:

        if metric in metrics.columns:

            delta_rows.append({
                "metric": metric,
                "baseline": base[metric],
                "rimgraph_v45": full[metric],
                "delta_full_minus_baseline":
                    full[metric] - base[metric]
            })

delta = pd.DataFrame(delta_rows)


# ============================================================
# 7. COMPLETION MARKERS
# ============================================================

completion_rows = []

for name, path in models.items():

    f = path / "COMPLETED.json"

    if f.exists():

        with open(f, "r") as fh:
            data = json.load(fh)

        completion_rows.append({
            "model": name,
            "completed_marker": True,
            "code_revision": data.get("code_revision")
        })

    else:
        completion_rows.append({
            "model": name,
            "completed_marker": False,
            "code_revision": None
        })

completion = pd.DataFrame(completion_rows)


# ============================================================
# PRINT EVERYTHING
# ============================================================

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

print("\n" + "="*100)
print("FINAL EXACT METRICS")
print("="*100)
print(metrics.to_string(index=False))

print("\n" + "="*100)
print("CONFUSION MATRICES")
print("="*100)
print(confusion.to_string(index=False))

print("\n" + "="*100)
print("BOOTSTRAP 95% CONFIDENCE INTERVALS")
print("="*100)
print(ci_table.to_string(index=False))

print("\n" + "="*100)
print("BASELINE → RIMGRAPH DELTAS")
print("="*100)
print(delta.to_string(index=False))

for name, df in calibration.items():
    print("\n" + "="*100)
    print(f"CALIBRATION CANDIDATES — {name}")
    print("="*100)
    print(df.to_string(index=False))

for name, df in histories.items():
    print("\n" + "="*100)
    print(f"TRAINING HISTORY — {name}")
    print("="*100)
    print(df.to_string(index=False))


# ============================================================
# SAVE CSV FILES
# ============================================================

metrics.to_csv(
    OUT / "01_exact_metrics.csv",
    index=False
)

confusion.to_csv(
    OUT / "02_confusion_matrices.csv",
    index=False
)

ci_table.to_csv(
    OUT / "03_bootstrap_95CI.csv",
    index=False
)

delta.to_csv(
    OUT / "04_baseline_vs_rimgraph_delta.csv",
    index=False
)

completion.to_csv(
    OUT / "05_completion_status.csv",
    index=False
)

for name, df in calibration.items():
    safe = name.lower().replace(" ", "_").replace("-", "_")
    df.to_csv(
        OUT / f"calibration_{safe}.csv",
        index=False
    )

for name, df in histories.items():
    safe = name.lower().replace(" ", "_").replace("-", "_")
    df.to_csv(
        OUT / f"history_{safe}.csv",
        index=False
    )


# ============================================================
# EXCEL WORKBOOK
# ============================================================

xlsx = OUT / "V45_ORIGA_COMPLETE_RESULTS.xlsx"

with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:

    metrics.to_excel(
        writer,
        sheet_name="Exact Metrics",
        index=False
    )

    confusion.to_excel(
        writer,
        sheet_name="Confusion Matrix",
        index=False
    )

    ci_table.to_excel(
        writer,
        sheet_name="Bootstrap CI",
        index=False
    )

    delta.to_excel(
        writer,
        sheet_name="Model Delta",
        index=False
    )

    completion.to_excel(
        writer,
        sheet_name="Completion",
        index=False
    )

    for name, df in calibration.items():
        sheet = (
            "Cal Baseline"
            if "Baseline" in name
            else "Cal RimGraph"
        )

        df.to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )

    for name, df in histories.items():
        sheet = (
            "Hist Baseline"
            if "Baseline" in name
            else "Hist RimGraph"
        )

        df.to_excel(
            writer,
            sheet_name=sheet,
            index=False
        )


print("\n" + "="*100)
print("✅ EXTRACTION COMPLETE")
print("="*100)

print("Folder:")
print(OUT)

print("\nExcel workbook:")
print(xlsx)


FINAL EXACT METRICS
    display_name      model_type held_out_source  seed  n_test    auroc  auroc_ci_low  auroc_ci_high    auprc  auprc_ci_low  auprc_ci_high  accuracy  balanced_accuracy  sensitivity  sensitivity_ci_low  sensitivity_ci_high  specificity  specificity_ci_low  specificity_ci_high  sensitivity_at_95_specificity  precision       f1      mcc    brier      nll      ece  threshold  tn  fp  fn  tp calibration_method  temperature  best_epoch  dice_disc  iou_disc  n_mask_disc  dice_cup  iou_cup  n_mask_cup
 Global Baseline global_baseline           ORIGA  2029     650 0.762683      0.718835       0.804552 0.521550      0.458488       0.590810  0.593846           0.675731     0.845238            0.791667             0.892857     0.506224            0.462656             0.556017                       0.279762   0.373684 0.518248 0.312236 0.195443 0.579452 0.180882   0.335965 244 238  26 142 robust_temperature     5.334482           5        NaN       NaN          NaN       NaN   

In [ ]:
# ================================================================
# RimGraph-DG V4.5 — HELD-OUT REFUGE
# ONE-CELL, RESUME-SAFE, T4 COLAB RUN
# ================================================================

GLAUCOMMA_OVERRIDES = {
    'manual_data_dir': '',
    'fast_dev_run': False,

    # ------------------------------------------------------------
    # REFUGE-specific V4.5 run
    # ------------------------------------------------------------
    'run_name': 'paper_run_v45_refuge',
    'code_revision': 'rimgraph-dg-v4.5-20260809',

    'seeds': [2029],

    # Train on ORIGA + G1020 → externally test REFUGE
    'fold_targets': ['REFUGE'],

    'run_global_baseline': True,
    'run_full_model': True,

    # May attempt V4.4 baseline reuse, but V4.5 will only accept
    # it when the exact split/config signatures are compatible.
    'baseline_reuse_run': 'paper_run_v44',

    'run_optuna': False,
    'optuna_trials': 8,

    'image_size': 320,
    'batch_size': 2,
    'grad_accum': 2,
    'fpn_dim': 128,
    'gradient_checkpointing': True,

    'baseline_epochs': 12,
    'full_epochs': 30,

    'seg_warmup_epochs': 5,
    'anatomy_warmup_epochs': 10,

    'num_workers': 0,
    'n_visual_examples': 2,

    # Critical for Colab interruption/reconnect
    'resume': True,
}


# ================================================================
# IMPORTS
# ================================================================

import hashlib
import json
import traceback
import urllib.request
from pathlib import Path

import torch


# ================================================================
# GPU CHECK
# ================================================================

print("=" * 80, flush=True)
print("RIMGRAPH-DG V4.5 — REFUGE HELD-OUT RUN", flush=True)
print("=" * 80, flush=True)

print("PyTorch:", torch.__version__, flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)

if not torch.cuda.is_available():
    raise RuntimeError(
        "T4 GPU is not active.\n"
        "Colab: Runtime > Change runtime type > T4 GPU, "
        "reconnect, then rerun this cell."
    )

print("GPU:", torch.cuda.get_device_name(0), flush=True)

if torch.cuda.get_device_properties(0).total_memory:
    gpu_gb = (
        torch.cuda.get_device_properties(0).total_memory
        / (1024 ** 3)
    )
    print(f"GPU memory: {gpu_gb:.2f} GB", flush=True)

print("=" * 80, flush=True)


# ================================================================
# IMMUTABLE V4.5 SOURCE
# ================================================================

COMMIT = 'd71c11be1a13a25ed0352cdde01453c17b234429'

EXPECTED_RAW_SHA256 = (
    '46ba27c7446662460456bc2bab186729'
    'c0df1b3e76533ce44f208150208335e2'
)

ROOT = (
    f'https://raw.githubusercontent.com/'
    f'AzizulHakim00/Glaucomma/{COMMIT}'
)

parts = [
    f'v4_parts/part_{i:02d}.py'
    for i in range(7)
]


# ================================================================
# DOWNLOAD BASE V4 RUNNER
# ================================================================

print("\n[LAUNCHER] Downloading immutable base runner...", flush=True)

raw_parts = []

for name in parts:
    url = f'{ROOT}/{name}'
    print(f'[LAUNCHER]   {name}', flush=True)

    source = urllib.request.urlopen(
        url
    ).read().decode('utf-8')

    raw_parts.append(source)

raw_code = '\n'.join(raw_parts)


# ================================================================
# RAW SOURCE INTEGRITY CHECK
# ================================================================

actual_raw = hashlib.sha256(
    raw_code.encode('utf-8')
).hexdigest()

print(
    "\n[LAUNCHER] Raw SHA256:",
    actual_raw,
    flush=True
)

assert actual_raw == EXPECTED_RAW_SHA256, (
    "Raw runner integrity check failed.\n"
    f"Expected: {EXPECTED_RAW_SHA256}\n"
    f"Actual:   {actual_raw}"
)

print("[LAUNCHER] Raw integrity check PASSED", flush=True)


# ================================================================
# APPLY V4.1 → V4.5 PATCH STACK
# ================================================================

patch_specs = [

    (
        'runner_patch_v41.py',
        'apply_v41'
    ),

    (
        'runner_patch_v42.py',
        'apply_v42'
    ),

    (
        'runner_patch_v43.py',
        'apply_v43'
    ),

    (
        'runner_patch_v43_autograd.py',
        'apply_v43_autograd'
    ),

    (
        'runner_patch_v44_runtime.py',
        'apply_v44_runtime'
    ),

    # V4.5 decoded-mask corrections
    (
        'runner_patch_v45_masks.py',
        'apply_v45_masks'
    ),

    (
        'runner_patch_v45_lowlabels.py',
        'apply_v45_lowlabels'
    ),
]


code = raw_code

for patch_name, function_name in patch_specs:

    print(
        f'[LAUNCHER] applying {patch_name}',
        flush=True
    )

    source = urllib.request.urlopen(
        f'{ROOT}/{patch_name}'
    ).read().decode('utf-8')

    namespace = {}

    exec(
        compile(
            source,
            patch_name,
            'exec'
        ),
        namespace,
        namespace
    )

    if function_name not in namespace:
        raise RuntimeError(
            f"{function_name} not found in "
            f"{patch_name}"
        )

    code = namespace[function_name](code)


# ================================================================
# VERIFY FINAL ASSEMBLED SCRIPT BEFORE EXECUTION
# ================================================================

compile(
    code,
    'rimgraph_dg_v45_refuge.py',
    'exec'
)

print(
    '\n[LAUNCHER] V4.5 REFUGE assembly PASSED',
    flush=True
)


# ================================================================
# EXPECTED OUTPUT
# ================================================================

DRIVE_ROOT = Path(
    '/content/drive/MyDrive/'
    'Glaucomma_RimGraphDG/'
    'paper_run_v45_refuge'
)

COMPLETION = (
    DRIVE_ROOT
    / 'RUN_COMPLETED.json'
)

FAILURE_TRACE = (
    DRIVE_ROOT
    / 'FAILURE_TRACEBACK.txt'
)

FAILURE_STATUS = (
    DRIVE_ROOT
    / 'FAILURE_STATUS.json'
)


print("\nExpected Drive run folder:")
print(DRIVE_ROOT)

print("\nExperimental design:")
print(
    "Train sources : ORIGA + G1020"
)
print(
    "Held-out test : REFUGE"
)
print(
    "Seed          : 2029"
)
print(
    "Version       : RimGraph-DG V4.5"
)


# ================================================================
# RUN
# ================================================================

try:

    exec(
        code,
        globals(),
        globals()
    )

    # ------------------------------------------------------------
    # Completion verification
    # ------------------------------------------------------------

    if not COMPLETION.exists():

        raise RuntimeError(
            "Runner returned without "
            "RUN_COMPLETED.json.\n"
            "The REFUGE experiment is therefore "
            "NOT considered complete."
        )

    print("\n" + "=" * 80)
    print(
        "✅ V4.5 REFUGE VERIFIED COMPLETION"
    )
    print("=" * 80)

    print(COMPLETION)

    # ------------------------------------------------------------
    # Show completion JSON
    # ------------------------------------------------------------

    try:
        completion_data = json.loads(
            COMPLETION.read_text(
                encoding='utf-8'
            )
        )

        print("\nRUN_COMPLETED.json:")
        print(
            json.dumps(
                completion_data,
                indent=2
            )
        )

    except Exception as e:
        print(
            "Could not display completion JSON:",
            e
        )

    # ------------------------------------------------------------
    # Show exact final metrics immediately
    # ------------------------------------------------------------

    fold_root = (
        DRIVE_ROOT
        / 'folds'
        / 'REFUGE'
        / 'seed_2029'
    )

    print("\n" + "=" * 80)
    print("FINAL HELD-OUT REFUGE RESULTS")
    print("=" * 80)

    result_paths = {

        "GLOBAL BASELINE":
            fold_root
            / 'global_baseline'
            / 'metrics.json',

        "RIMGRAPH-DG V4.5":
            fold_root
            / 'rimgraph_v4'
            / 'metrics.json',
    }

    final_results = {}

    for model_name, metric_path in result_paths.items():

        print(
            f"\n--- {model_name} ---"
        )

        if not metric_path.exists():

            print(
                "❌ metrics.json missing:",
                metric_path
            )

            continue

        metrics = json.loads(
            metric_path.read_text(
                encoding='utf-8'
            )
        )

        final_results[model_name] = metrics

        preferred_keys = [

            'held_out_source',
            'seed',
            'n_test',

            'auroc',
            'auroc_ci_low',
            'auroc_ci_high',

            'auprc',
            'auprc_ci_low',
            'auprc_ci_high',

            'accuracy',
            'balanced_accuracy',

            'sensitivity',
            'sensitivity_ci_low',
            'sensitivity_ci_high',

            'specificity',
            'specificity_ci_low',
            'specificity_ci_high',

            'sensitivity_at_95_specificity',

            'precision',
            'f1',
            'mcc',

            'brier',
            'nll',
            'ece',

            'threshold',

            'tn',
            'fp',
            'fn',
            'tp',

            'calibration_method',
            'temperature',

            'best_epoch',

            'dice_disc',
            'iou_disc',
            'n_mask_disc',

            'dice_cup',
            'iou_cup',
            'n_mask_cup',
        ]

        for key in preferred_keys:

            if key in metrics:

                print(
                    f"{key:32s}: "
                    f"{metrics[key]}"
                )


    # ============================================================
    # BASELINE VS RIMGRAPH DELTAS
    # ============================================================

    if (
        "GLOBAL BASELINE"
        in final_results
        and
        "RIMGRAPH-DG V4.5"
        in final_results
    ):

        base = final_results[
            "GLOBAL BASELINE"
        ]

        full = final_results[
            "RIMGRAPH-DG V4.5"
        ]

        print("\n" + "=" * 80)
        print(
            "REFUGE: RIMGRAPH − BASELINE DELTAS"
        )
        print("=" * 80)

        compare_metrics = [

            'auroc',
            'auprc',
            'accuracy',
            'balanced_accuracy',

            'sensitivity',
            'specificity',

            'precision',
            'f1',
            'mcc',

            'brier',
            'nll',
            'ece',
        ]

        for metric in compare_metrics:

            if (
                metric in base
                and metric in full
            ):

                delta = (
                    float(full[metric])
                    - float(base[metric])
                )

                print(
                    f"{metric:24s}: "
                    f"{delta:+.6f}"
                )


    print("\n" + "=" * 80)
    print("IMPORTANT")
    print("=" * 80)

    print(
        "This completion represents ONLY the "
        "held-out REFUGE V4.5 fold."
    )

    print(
        "Do not combine it with ORIGA until "
        "the exact final metrics are extracted."
    )

    print(
        "After REFUGE, the remaining primary "
        "held-out fold is G1020."
    )


# ================================================================
# FAILURE HANDLING
# ================================================================

except BaseException:

    trace = traceback.format_exc()

    print(
        '\n=== V4.5 REFUGE FAILURE TRACEBACK ===',
        flush=True
    )

    print(
        trace,
        flush=True
    )

    # Save local copy
    try:

        Path(
            '/content/'
            'RimGraph_V45_REFUGE_'
            'FAILURE_TRACEBACK.txt'
        ).write_text(
            trace,
            encoding='utf-8'
        )

    except Exception:
        pass

    # Save Drive copy
    try:

        DRIVE_ROOT.mkdir(
            parents=True,
            exist_ok=True
        )

        FAILURE_TRACE.write_text(
            trace,
            encoding='utf-8'
        )

        FAILURE_STATUS.write_text(

            json.dumps(
                {
                    'status': 'failed',
                    'run_name':
                        'paper_run_v45_refuge',
                    'held_out_source':
                        'REFUGE',
                    'seed': 2029,
                    'code_revision':
                        'rimgraph-dg-v4.5-20260809',
                    'traceback_file':
                        str(FAILURE_TRACE),
                },
                indent=2
            ),

            encoding='utf-8'
        )

        print(
            "\nFailure information saved to:"
        )

        print(
            FAILURE_TRACE
        )

    except Exception as drive_error:

        print(
            "Could not persist failure "
            "information to Drive:",
            drive_error,
            flush=True
        )

    raise

RIMGRAPH-DG V4.5 — REFUGE HELD-OUT RUN
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB

[LAUNCHER] Downloading immutable base runner...
[LAUNCHER]   v4_parts/part_00.py
[LAUNCHER]   v4_parts/part_01.py
[LAUNCHER]   v4_parts/part_02.py
[LAUNCHER]   v4_parts/part_03.py
[LAUNCHER]   v4_parts/part_04.py
[LAUNCHER]   v4_parts/part_05.py
[LAUNCHER]   v4_parts/part_06.py

[LAUNCHER] Raw SHA256: 46ba27c7446662460456bc2bab186729c0df1b3e76533ce44f208150208335e2
[LAUNCHER] Raw integrity check PASSED
[LAUNCHER] applying runner_patch_v41.py
[LAUNCHER] applying runner_patch_v42.py
[LAUNCHER] applying runner_patch_v43.py
[LAUNCHER] applying runner_patch_v43_autograd.py
[LAUNCHER] applying runner_patch_v44_runtime.py
[LAUNCHER] applying runner_patch_v45_masks.py
[LAUNCHER] applying runner_patch_v45_lowlabels.py

[LAUNCHER] V4.5 REFUGE assembly PASSED

Expected Drive run folder:
/content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v45_refuge

Experimental design:
Train so

## RimGraph-DG V4 configuration

,setting,value
0,kaggle_dataset,arnavjain1/glaucoma-datasets
1,manual_data_dir,
2,sources,"['ORIGA', 'REFUGE', 'G1020']"
3,fold_targets,['REFUGE']
4,image_size,320
5,num_workers,0
6,canonicalize_laterality,True
7,exclude_cross_source_duplicates,True
8,project_name,RimGraph_DG_V4
9,run_name,paper_run_v45_refuge


## Downloading or locating Kaggle dataset

Using Colab cache for faster access to the 'glaucoma-datasets' dataset.
Dataset root: /kaggle/input/glaucoma-datasets


...

,source,total,valid_disc,valid_cup,valid_vcdr,disc_valid_rate,cup_valid_rate,vcdr_valid_rate
0,G1020,1020,1020,790,790,1.0,0.7745,0.7745
1,ORIGA,650,650,650,650,1.0,1.0000,1.0000
2,REFUGE,800,800,800,800,1.0,1.0000,1.0000


V4.5 DECODED MASK AUDIT: PASSED
[PREFLIGHT] constructing full RimGraph model ...
[BACKBONE] loading convnext_tiny.fb_in22k_ft_in1k pretrained=True ...


[BACKBONE] ready: convnext_tiny.fb_in22k_ft_in1k
[PREFLIGHT] full-model forward/backward PASSED | peak allocated=0.35 GB
[V4.5] RimGraph checkpoints from earlier revisions are intentionally NOT reused.


## Seed 2029 — held-out REFUGE

[BASELINE REUSE] prior artifacts incomplete in /content/drive/MyDrive/Glaucomma_RimGraphDG/paper_run_v44/folds/REFUGE/seed_2029/global_baseline; training baseline normally
[BACKBONE] loading convnext_tiny.fb_in22k_ft_in1k pretrained=True ...
[BACKBONE] ready: convnext_tiny.fb_in22k_ft_in1k
[BASELINE] epoch 1/12 start | batches=668
[BASELINE] epoch 1 batch 1/668
[BASELINE] epoch 1 batch 167/668
